In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/TriviaQA_UND_gpt4o_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_TriviaQA_UND_qa_gpt_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

In [6]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/TriviaQA_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 198.34ba/s]


577281

## Rewriting with Gemini
GPT-4o rewriting, then GPT-4o QA later

In [7]:
from helper_functions_qr import modification_in_batch

In [8]:
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"
input_file = "./intermediate/TriviaQA_UND_gpt4o_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl"



In [9]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'normalized_aliases', client, model)

Total samples to process: 159
Batch size: 3


Processing batches:  72%|███████▏  | 38/53 [18:38<07:36, 30.40s/it]

Error processing sample 114: Invalid \escape: line 3 column 36 (char 111)


Processing batches: 100%|██████████| 53/53 [27:52<00:00, 31.55s/it]


All batch processing completed! Total processed: 159 samples
Results saved to: ./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl


In [10]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,"""Which Gilbert and Sullivan operetta is sub ti...",Which Gilbert and Sullivan operetta is famousl...,"[ruddigore, ruddigore witch s curse, rudigore,...",[Ruddigore],The query seeks to identify a specific Gilbert...,1.000000,1,1.00
1,Wolf Mankowitz wrote the 1953 novel ‘A Kid For...,What word completes the title of Wolf Mankowit...,"[farthings, farthing, farthing disambiguation,...",[Farthings],The query contains multiple issues that make i...,1.000000,1,1.00
2,Who is the patron saint of dancers?,Which saints are often invoked as patron saint...,"[saint crescentia, saint vitus, vitus, s vito,...",[Saint Vitus],The query asks for a specific individual (a pa...,1.000000,1,1.00
3,"When Mr Benn was looking for an adventure, wha...","In the children's animated series *Mr Benn*, w...",[fancy dress shop],[A costume shop],"The query references 'Mr Benn,' a name that do...",0.400000,0,0.75
4,What was made and repaired by a Wainwright?,What types of items were traditionally made an...,"[wagon, front axle assembly, wagon vehicle, ho...",[Wagons],"The query references 'a Wainwright,' which is ...",1.000000,1,1.00
...,...,...,...,...,...,...,...,...
154,"Which mammal has species called 'leopard', 'Gr...",Which elite military special operations force ...,"[american navy sea air and land teams, us navy...",[Seal],The query asks for a single mammal that encomp...,1.000000,1,1.00
155,"Which preparation still in use today, was know...","Which oral hygiene preparation, known in 4th c...","[toothpaste tube, toothpaste, toofpaste, tube ...",[Toothpaste],The query seeks a preparation historically lin...,1.000000,1,1.00
156,Dr. Benjamin Rush gave what expeditionary grou...,Dr. Benjamin Rush gave what expeditionary grou...,"[louis and clark expidition, louis and clark e...",[Lewis and Clark Expedition],The query requires identifying a specific hist...,1.000000,1,1.00
157,"Which rust free cars were built in Dunmurry, N...","Which car, notable for its corrosion-resistant...","[de lorean, delorean, deloreans, de loreans, d...",[DeLorean DMC-12],"The query contains geographic (Dunmurry, North...",0.666667,0,1.00


## Modified queries QA using GPT-4o

### Loading modified data

In [11]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 159 examples [00:00, 5548.67 examples/s]


### Implementation

In [12]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [13]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 16/16 [02:26<00:00,  9.14s/it]


In [14]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 266.81ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,"""Which Gilbert and Sullivan operetta is sub ti...",Which Gilbert and Sullivan operetta is famousl...,"[ruddigore, ruddigore witch s curse, rudigore,...",[Ruddigore],The query seeks to identify a specific Gilbert...,1.000000,1,1.00,[Ruddigore]
1,Wolf Mankowitz wrote the 1953 novel ‘A Kid For...,What word completes the title of Wolf Mankowit...,"[farthings, farthing, farthing disambiguation,...",[Farthings],The query contains multiple issues that make i...,1.000000,1,1.00,[Farthings]
2,Who is the patron saint of dancers?,Which saints are often invoked as patron saint...,"[saint crescentia, saint vitus, vitus, s vito,...",[Saint Vitus],The query asks for a specific individual (a pa...,1.000000,1,1.00,[Saint Vitus]
3,"When Mr Benn was looking for an adventure, wha...","In the children's animated series *Mr Benn*, w...",[fancy dress shop],[A costume shop],"The query references 'Mr Benn,' a name that do...",0.400000,0,0.75,[A costume shop]
4,What was made and repaired by a Wainwright?,What types of items were traditionally made an...,"[wagon, front axle assembly, wagon vehicle, ho...",[Wagons],"The query references 'a Wainwright,' which is ...",1.000000,1,1.00,"[Wagons, Carts, Carriages, Wheels, Axles]"
...,...,...,...,...,...,...,...,...,...
154,"Which mammal has species called 'leopard', 'Gr...",Which elite military special operations force ...,"[american navy sea air and land teams, us navy...",[Seal],The query asks for a single mammal that encomp...,1.000000,1,1.00,[Spetsnaz (Russian Special Forces)]
155,"Which preparation still in use today, was know...","Which oral hygiene preparation, known in 4th c...","[toothpaste tube, toothpaste, toofpaste, tube ...",[Toothpaste],The query seeks a preparation historically lin...,1.000000,1,1.00,[Toothpaste]
156,Dr. Benjamin Rush gave what expeditionary grou...,Dr. Benjamin Rush gave what expeditionary grou...,"[louis and clark expidition, louis and clark e...",[Lewis and Clark Expedition],The query requires identifying a specific hist...,1.000000,1,1.00,[Lewis and Clark Expedition]
157,"Which rust free cars were built in Dunmurry, N...","Which car, notable for its corrosion-resistant...","[de lorean, delorean, deloreans, de loreans, d...",[DeLorean DMC-12],"The query contains geographic (Dunmurry, North...",0.666667,0,1.00,[- DeLorean DMC-12]


## Evaluations

### Squad EM+F1

In [15]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_TriviaQA_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 159 examples [00:00, 53338.75 examples/s]


In [16]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_Gemini_TriviaQA_UND_gpt4o_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_Gemini_TriviaQA_UND_gpt4o_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 426.08ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,"""Which Gilbert and Sullivan operetta is sub ti...",Which Gilbert and Sullivan operetta is famousl...,"[ruddigore, ruddigore witch s curse, rudigore,...",[Ruddigore],The query seeks to identify a specific Gilbert...,1.000000,1,1.00,[Ruddigore],1,1.000000
1,Wolf Mankowitz wrote the 1953 novel ‘A Kid For...,What word completes the title of Wolf Mankowit...,"[farthings, farthing, farthing disambiguation,...",[Farthings],The query contains multiple issues that make i...,1.000000,1,1.00,[Farthings],1,1.000000
2,Who is the patron saint of dancers?,Which saints are often invoked as patron saint...,"[saint crescentia, saint vitus, vitus, s vito,...",[Saint Vitus],The query asks for a specific individual (a pa...,1.000000,1,1.00,[Saint Vitus],1,1.000000
3,"When Mr Benn was looking for an adventure, wha...","In the children's animated series *Mr Benn*, w...",[fancy dress shop],[A costume shop],"The query references 'Mr Benn,' a name that do...",0.400000,0,0.75,[A costume shop],0,0.400000
4,What was made and repaired by a Wainwright?,What types of items were traditionally made an...,"[wagon, front axle assembly, wagon vehicle, ho...",[Wagons],"The query references 'a Wainwright,' which is ...",1.000000,1,1.00,"[Wagons, Carts, Carriages, Wheels, Axles]",1,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
154,"Which mammal has species called 'leopard', 'Gr...",Which elite military special operations force ...,"[american navy sea air and land teams, us navy...",[Seal],The query asks for a single mammal that encomp...,1.000000,1,1.00,[Spetsnaz (Russian Special Forces)],0,0.285714
155,"Which preparation still in use today, was know...","Which oral hygiene preparation, known in 4th c...","[toothpaste tube, toothpaste, toofpaste, tube ...",[Toothpaste],The query seeks a preparation historically lin...,1.000000,1,1.00,[Toothpaste],1,1.000000
156,Dr. Benjamin Rush gave what expeditionary grou...,Dr. Benjamin Rush gave what expeditionary grou...,"[louis and clark expidition, louis and clark e...",[Lewis and Clark Expedition],The query requires identifying a specific hist...,1.000000,1,1.00,[Lewis and Clark Expedition],1,1.000000
157,"Which rust free cars were built in Dunmurry, N...","Which car, notable for its corrosion-resistant...","[de lorean, delorean, deloreans, de loreans, d...",[DeLorean DMC-12],"The query contains geographic (Dunmurry, North...",0.666667,0,1.00,[- DeLorean DMC-12],0,0.666667


In [17]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 73.58
New answers after modification F1 Score (avg): 83.63
Original answers Exact Match (avg): 64.78
Original answers F1 Score (avg): 75.77
F1: t=2.026, p=0.0437
EM: t=1.703, p=0.0896


### Ragas AA

In [18]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [19]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_Gemini_TriviaQA_UND_gpt4o_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_Gemini_TriviaQA_UND_gpt4o_all_new_scores.csv")

Generating train split: 159 examples [00:00, 50869.13 examples/s]
Calculating short answer accuracy:   3%|▎         | 4/159 [00:12<08:14,  3.19s/it]

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 68.85ba/s]


175573

In [20]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 84.28
modified AA (avg): 92.77
AA: t=2.505, p=0.0128


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_Gemini_TriviaQA_UND_gpt4o_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.43s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 159
Generation complete: 159 prompts
Average prompt length: 444 bytes (~111 tokens)

Analyze the following input user query:

{"query": "Which Gilbert and Sullivan operetta is famously associated with 'The Witches Curse'?"}

Please provide your analysis in the following JSON format:

{"query": "Which Gilbert and Sullivan operetta is famously associated with 'The Witches Curse'?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 32/32 [24:18<00:00, 45.58s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,"""Which Gilbert and Sullivan operetta is sub ti...",Which Gilbert and Sullivan operetta is famousl...,['ruddigore' 'ruddigore witch s curse' 'rudigo...,['Ruddigore'],The query seeks to identify a specific Gilbert...,1.000000,1,1.00,['Ruddigore'],1,1.000000,1.00,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Which Gilbert and Sullivan oper...",underspecified
1,Wolf Mankowitz wrote the 1953 novel ‘A Kid For...,What word completes the title of Wolf Mankowit...,['farthings' 'farthing' 'farthing disambiguati...,['Farthings'],The query contains multiple issues that make i...,1.000000,1,1.00,['Farthings'],1,1.000000,1.00,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""What word completes the title o...",fully specified
2,Who is the patron saint of dancers?,Which saints are often invoked as patron saint...,['saint crescentia' 'saint vitus' 'vitus' 's v...,['Saint Vitus'],The query asks for a specific individual (a pa...,1.000000,1,1.00,['Saint Vitus'],1,1.000000,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""Which saints are often invoked ...",fully specified
3,"When Mr Benn was looking for an adventure, wha...","In the children's animated series *Mr Benn*, w...",['fancy dress shop'],['A costume shop'],"The query references 'Mr Benn,' a name that do...",0.400000,0,0.75,['A costume shop'],0,0.400000,0.75,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""In the children's animated seri...",fully specified
4,What was made and repaired by a Wainwright?,What types of items were traditionally made an...,['wagon' 'front axle assembly' 'wagon vehicle'...,['Wagons'],"The query references 'a Wainwright,' which is ...",1.000000,1,1.00,['Wagons' 'Carts' 'Carriages' 'Wheels' 'Axles'],1,1.000000,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What types of items were tradit...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,"Which mammal has species called 'leopard', 'Gr...",Which elite military special operations force ...,['american navy sea air and land teams' 'us na...,['Seal'],The query asks for a single mammal that encomp...,1.000000,1,1.00,['Spetsnaz (Russian Special Forces)'],0,0.285714,0.50,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""Which elite military special op...",underspecified
155,"Which preparation still in use today, was know...","Which oral hygiene preparation, known in 4th c...",['toothpaste tube' 'toothpaste' 'toofpaste' 't...,['Toothpaste'],The query seeks a preparation historically lin...,1.000000,1,1.00,['Toothpaste'],1,1.000000,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""Which oral hygiene preparation,...",fully specified
156,Dr. Benjamin Rush gave what expeditionary grou...,Dr. Benjamin Rush gave what expeditionary grou...,['louis and clark expidition' 'louis and clark...,['Lewis and Clark Expedition'],The query requires identifying a specific hist...,1.000000,1,1.00,['Lewis and Clark Expedition'],1,1.000000,1.00,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Dr. Benjamin Rush gave what e...",underspecified
157,"Which rust free cars were built in Dunmurry, N...","Which car, notable for its corrosion-resistant...",['de lorean' 'delorean' 'deloreans' 'de lorean...,['DeLorean DMC-12'],"The query contains geographic (Dunmurry, North...",0.666667,0,1.00,['- DeLorean DMC-12'],0,0.666667,1.00,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""Which car, notable for its corr...",fully specified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.836478
underspecified     0.163522
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    133
underspecified      26
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/TriviaQA_UND_Gemini_rewritten_reclassified.csv')